# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [1]:
import os

# Async CUDA allocator
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# If cuDNN autotune fails, fall back to a safe (but slower) algorithm.
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true' 

### 1.2. Imports

In [2]:
from _imports import * # Centralized file containing all imports

2025-06-18 09:25:18.677865: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-18 09:25:18.691683: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750249518.708101  246067 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750249518.713050  246067 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-18 09:25:18.730072: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### 1.3. GPU Management

In [3]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()


TensorFlow GPU Monitor - 2025-06-18 09:25:20
TensorFlow Configuration
Version        : 2.18.0
CUDA Support   : Yes
CUDA Version   : 12.5.1
cuDNN Version  : 9
GPUs Detected  : 1
Default Device : /device:GPU:0

GPU Information
GPU Name                      Memory Usage         Temp   Util  
--------------------------------------------------------------------------------
0   NVIDIA GeForce RTX 3070      1.4GB /    8.0GB  50C    20%   

Memory Summary
Total GPU Memory :      8.0 GB
Used Memory      :      1.4 GB ( 17.9%)
Free Memory      :      6.6 GB ( 82.1%)



2025-06-18 09:25:20.432356: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1750249520.432379  246067 gpu_process_state.cc:201] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1750249520.433733  246067 gpu_device.cc:2022] Created device /device:GPU:0 with 4838 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:b3:00.0, compute capability: 8.6


## 2. Run Parameters 

In [4]:
NUM_TRIALS = 10
EPOCHS = 1

In [5]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

In [6]:
TOP_K = 1 # Number of top trials to save

# True -> the greatest, the better
# False -> the least, the better
RANK_DESCENDING = False  

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "best_val_accuracy"

In [7]:
# Set to an existing path to resume training
# Set to None to start a new run
RESUME_TRAINING_PATH = "runs/nas_seed_v0.0"
RUN_DIR = RESUME_TRAINING_PATH or create_run_directory(prefix="nas_")

## 3. Getters

### 3.1. Callbacks

In [8]:
def get_callbacks(trial: optuna.Trial, backup_dir: str) -> List[tf.keras.callbacks.Callback]:
    """
    Constructs and returns a list of Keras callbacks tailored for Optuna trials.

    Args:
        trial (optuna.Trial): The current Optuna trial object.
        backup_dir (str): Directory where the backup files will be stored.

    Returns:
        List[tf.keras.callbacks.Callback]: A list of callbacks to pass into `model.fit()`.
    """
    # Metric to monitor for early stopping and checkpointing
    monitor: str = "val_loss"

    # Stop training early if no improvement in validation loss for N epochs
    early_stopping = callbacks.EarlyStopping(
        monitor=monitor,
        patience=10,  # number of epochs to wait
        restore_best_weights=True,
        verbose=1,
    )

    # Reduce learning rate if validation loss plateaus
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        patience=5,  # how many epochs to wait before reducing LR
        factor=0.2,  # reduce LR by this factor
        min_lr=1e-6,  # don't reduce below this
        verbose=1,
    )
    
    # Backup and restore the model
    # backup = callbacks.BackupAndRestore(backup_dir=backup_dir)
    
    # Model checkpointing
    # checkpoint = callbacks.ModelCheckpoint(
    #     filepath=os.path.join(backup_dir, "checkpoint.h5"),
    #     monitor=monitor,
    #     save_best_only=True,
    #     save_weights_only=True,
    # )

    #! ——————— WARNING: the callbacks below do not work with multi-objective —————— !#
    # Custom callback to prune trial if NaN loss is encountered
    nan_pruner_callback = callbacks.TerminateOnNaN()

    # Optuna's built-in pruning callback for early trial termination
    pruning_callback = KerasPruningCallback(trial, monitor, interval=5)
    #! ———————————————————————————————————————————————————————————————————————————— !#

    # Return the complete list of callbacks
    return [early_stopping, reduce_lr, nan_pruner_callback, pruning_callback]


def get_activation(function: str) -> tf.keras.layers.Layer:
    """
    Returns the activation layer based on the provided function name.
    
    Args:
        function (str): Name of the activation function.
        
    Returns:
        tf.keras.layers.Layer: Corresponding activation layer.
    """
    if function == "relu":
        return layers.Activation("relu")
    elif function == "tanh":
        return layers.Activation("tanh")
    elif function == "sigmoid":
        return layers.Activation("sigmoid")
    elif function == "swish":
        return layers.Activation("swish")
    else:
        raise ValueError(f"Unsupported activation function: {function}")

## 4. Hyperparameters

In [9]:
hparams = HParams(
    activation_choices=[
        #? ReLU family
        "relu",
        # "leaky_relu",
        # "elu",
        # "celu",
        # "selu",
        #? Smooth ReLU-like and modern variants
        # "softplus",
        # "gelu",
        # "mish",
        "swish",
        # "hard_silu",
        #? Tanh family
        # "tanh",
        # "hard_tanh",
        # "softsign",
        #? Sigmoid family
        # "sigmoid",
        # "hard_sigmoid",
        # "log_sigmoid",
        #? Linear and Exponential
        # "linear",
        # "exponential",
        #? Gated and transformer-related
        # "glu",
        # "softmax",
        #? Sparsity and uncommon
        # "hard_shrink",
    ],
    regularizer_choices=[
        "none",
        "l1",
        "l2",
        "l1l2",
    ],
    optimizer_choices=[
        # "SGD",
        # "RMSprop",
        # "Adam",
        # "AdamW",
        # "Adadelta",
        # "Adagrad",
        # "Adamax",
        # "Adafactor",
        # "Nadam",
        # "Ftrl",
        "Lion",
        # "Lamb",
        # "LossScaleOptimizer",
    ],
    scaler_choices=[
        "StandardScaler",
        "MinMaxScaler_0_1",
        "MinMaxScaler_-1_1",
        "RobustScaler",
        "QuantileTransformer",
        "PowerTransformer",
    ],
    l1_value=1e-2,
    l2_value=1e-2,
    min_lr=8e-5,
    max_lr=2e-4,
    
    # Fixed learning rate
    lr_value=1e-4,
)

initializer_options = [
    initializers.Zeros(),
    initializers.Ones(),
    initializers.Constant(),
    initializers.RandomNormal(),
    initializers.RandomUniform(),
    initializers.TruncatedNormal(),
    initializers.GlorotNormal(),
    initializers.GlorotUniform(),
    initializers.HeNormal(),
    initializers.HeUniform(),
    initializers.LecunNormal(),
    initializers.LecunUniform(),
    initializers.Identity(),
    initializers.Orthogonal(),
    initializers.VarianceScaling(),
]

## 5. Objective Function

In [10]:
def objective(
    trial: optuna.Trial,
    backup_dir: str,
    model_dir: str,
    fig_dir: str,
    logs_dir: str,
    history_dir: str,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    show_summary: bool = False,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        X (List[np.ndarray]): List of input arrays.
        y (List[np.ndarray]): List of label arrays.
        backup_dir (str): Path to store backup files.
        model_dir (str): Path to store full models.
        fig_dir (str): Path to store plots.
        logs_dir (str): Path to store logs.
        history_dir (str): Path to store training history.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        show_summary (bool): If True, display the model summary.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """
    (clear_session(), gc.collect())

    # ——————————————————————————————————— Setup —————————————————————————————————— #
    seed = trial.suggest_int("seed", 0, 1000, step=1)

    # Loading dataset
    print("Generating random training data...")
    num_samples = 10000
    input_shape = (32, 32, 3)  # RGB images
    num_classes = 10

    X_dataset = np.random.randn(num_samples, *input_shape).astype(np.float32)
    y_dataset = tf.keras.utils.to_categorical(np.random.randint(0, num_classes, num_samples), num_classes)

    # Split the dataset into training, validation using sklearn
    X_train, X_val, y_train, y_val = train_test_split(X_dataset, y_dataset, test_size=0.2, random_state=seed)

    print(f"Training data shape: {X_train.shape}")
    print(f"Training labels shape: {y_train.shape}")

    # —————————————————————————————— Training seeds —————————————————————————————— #
    t_seed = 0

    np.random.seed(t_seed)
    tf.random.set_seed(t_seed)

    # ———————————————————————————————————————————————————————————————————————————— #

    model = None
    try:
        # ———————————————————————————————————————————————————————————————————————————— #
        #                               Model Contruction                              #
        # ———————————————————————————————————————————————————————————————————————————— #
        initializer = tf.keras.initializers.GlorotUniform(seed=t_seed)

        # ——————————————————————————————————— Input —————————————————————————————————— #
        inputs = layers.Input(shape=input_shape, name="image_input")

        # ——————————————————————————— Convolutional layers ——————————————————————————— #
        x = layers.Conv2D(
            32,
            (3, 3),
            activation="relu",
            kernel_initializer=initializer,
        )(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)

        x = layers.Conv2D(
            64,
            (3, 3),
            activation="relu",
            kernel_initializer=initializer,
        )(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)

        x = layers.Conv2D(
            128,
            (3, 3),
            activation="relu",
            kernel_initializer=initializer,
        )(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)

        # ——————————————————————————————— Dense layers ——————————————————————————————— #
        x = layers.Flatten()(x)
        x = layers.Dense(
            256,
            activation="relu",
            kernel_initializer=initializer,
        )(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(
            128,
            activation="relu",
            kernel_initializer=initializer,
        )(x)
        x = layers.Dropout(0.3)(x)

        # —————————————————————————————————— Output —————————————————————————————————— #
        outputs = layers.Dense(num_classes, activation="softmax", kernel_initializer=initializer,)(x)

        # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
        model = Model(inputs=inputs, outputs=(outputs,))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                                Train the Model                               #
        # ———————————————————————————————————————————————————————————————————————————— #
        model.summary() if show_summary else None

        model.compile(
            optimizer=hparams.get_optimizer(trial),
            loss=losses.CategoricalCrossentropy(),
            metrics=["accuracy"],
        )

        batch_size = 64
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks(trial, backup_dir),
            verbose=2,
        )

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss = min(history.history["val_loss"])

        if size_penalizer == "flops":
            loss = compute_flops_penalized_loss(loss=loss, model=model)
        elif size_penalizer == "params":
            loss = compute_params_penalized_loss(loss=loss, model=model, params_penalty_factor=1e-8)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ——————————————————————————————— Plot Results ——————————————————————————————— #
        # Configure axis
        epochs = list(range(1, len(history.history["loss"]) + 1))
        train_loss = history.history["loss"]
        val_loss = history.history["val_loss"]
        train_acc = history.history.get("accuracy", [])
        val_acc = history.history.get("val_accuracy", [])
        val_loss_best = min(history.history["val_loss"])

        # Create figure with two subplots
        fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Loss
        ax_loss.plot(epochs, train_loss, marker="o", linestyle="-", label="Training Loss")
        ax_loss.plot(epochs, val_loss, marker="x", linestyle="--", label="Validation Loss")
        ax_loss.set_title("Training & Validation Loss")
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Loss")
        ax_loss.set_xticks(epochs)
        ax_loss.set_ylim(0, max(max(train_loss), max(val_loss)) * 1.05)
        ax_loss.grid(True)
        ax_loss.legend()

        # Right: Accuracy (if available)
        if train_acc and val_acc:
            ax_acc.plot(epochs, train_acc, marker="v", linestyle="-", label="Training Accuracy")
            ax_acc.plot(epochs, val_acc, marker="^", linestyle="--", label="Validation Accuracy")
            ax_acc.set_title("Training & Validation Accuracy")
            ax_acc.set_xlabel("Epoch")
            ax_acc.set_ylabel("Accuracy")
            ax_acc.set_xticks(epochs)
            ax_acc.set_ylim(0, 1)
            ax_acc.grid(True)
            ax_acc.legend()

            trial.set_user_attr("best_train_accuracy", float(max(train_acc)))
            trial.set_user_attr("best_val_accuracy", float(max(val_acc)))
        else:
            ax_acc.axis("off")  # hide if accuracy not present

        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, f"trial_{trial.number}.png"), dpi=300)
        plt.close(fig)

        # ———————————————————————— Save model characteristics ———————————————————————— #
        params = model.count_params()
        bits_per_param = tf.dtypes.as_dtype(POLICY.variable_dtype).size
        peak_mem_usage, inference_time = get_memory_and_time(
            model,
            batch_size=batch_size,
            device="GPU:0",
            warmup_runs=10,
            test_runs=20,
        )

        trial.set_user_attr("num_params", params)
        trial.set_user_attr("model_size", params * bits_per_param)
        trial.set_user_attr("flops", get_flops(model))
        trial.set_user_attr("macs", get_macs(model))
        trial.set_user_attr("model_summary", capture_model_summary(model))
        trial.set_user_attr("peak_memory_usage", peak_mem_usage)
        trial.set_user_attr("inference_time", inference_time)

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ——————————————————————————————— Save history ——————————————————————————————— #
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }

        # Add accuracy metrics if available
        if "accuracy" in history.history:
            history_data["train_accuracy"] = history.history["accuracy"]
        if "val_accuracy" in history.history:
            history_data["val_accuracy"] = history.history["val_accuracy"]

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # ———————————————————————————————————————————————————————————————————————————— #
        return loss

    except optuna.exceptions.TrialPruned:
        raise  # simply propagate pruning
    except tf.errors.ResourceExhaustedError as oom_err:
        print(f"\n❌ Trial {trial.number} hit OOM (Resource Exhausted)\n")
        with open(os.path.join(logs_dir, f"oom_trials.log"), "a") as f:
            f.write(f"OOM error during trial {trial.number}:\n{traceback.format_exc()}\n\n")

        return float("inf")  # Return bad loss
    except Exception as e:
        with open(os.path.join(logs_dir, f"error_trial_{trial.number}.log"), "w") as f:
            f.write(f"An error occurred during trial:\n{e}\n{traceback.format_exc()}\n\n")

        raise  # Re-raise the exception to propagate it
    finally:
        for v in [
            "model",
            "history",
            "history_df",
        ]:
            if v in globals() and globals()[v] is not None:
                del globals()[v]
        (plt.cla(), plt.clf(), plt.close("all"))

## 7. Code Health Check

In [11]:
# resources_dir = os.path.join(RUN_DIR, "resources")
# os.makedirs(resources_dir, exist_ok=True)
# log_resources(log_dir=resources_dir)

## Main

In [12]:
try:
    # ———————————————————————————————— Study Setup ——————————————————————————————— #
    # Initialize directories for the study
    (
        study_dir,
        args_dir,
        fig_dir,
        backup_dir,
        history_dir,
        model_dir,
        logs_dir,
    ) = init_study_dirs(RUN_DIR)

    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=f"sqlite:///{study_dir}/optuna_study.db",
        direction="minimize",
        pruner=optuna.pruners.HyperbandPruner(),
        load_if_exists=True,
    )

    study.optimize(
        lambda trial: objective(
            trial,
            backup_dir=backup_dir,
            model_dir=model_dir,
            fig_dir=fig_dir,
            logs_dir=logs_dir,
            history_dir=history_dir,
            epochs=EPOCHS,
            size_penalizer=None,
            show_summary=False,
        ),
        n_trials=get_remaining_trials(study, NUM_TRIALS),
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
        n_jobs=1,  # If you have multiple GPUs/Cores
        show_progress_bar=False,
    )

    # ——————————————————————— Processing the Study Results ——————————————————————— #
    top_trials = get_top_trials(
        study,
        top_k=TOP_K,
        rank_key=RANK_KEY,
        rank_descending=RANK_DESCENDING,
    )

    cleanup_paths = [
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
        (history_dir, "trial_{trial_id}.csv"),
    ]

    rename_paths = [
        (model_dir, ".keras"),
        (fig_dir, ".png"),
        (history_dir, ".csv"),
    ]

    extra_attrs = [
        "best_train_accuracy",
        "best_val_accuracy",
    ]

    save_top_k_trials(
        top_trials,
        args_dir=args_dir,
        study=study,
        extra_attrs=extra_attrs,
    )
    cleanup_non_top_trials(
        {t.number for t in study.trials},  # All trials
        {t.number for t in top_trials},  # Top trials ids
        cleanup_paths,
    )
    rename_top_k_files(top_trials, rename_paths)

    # ————————————————————————————— Log Trial Results ———————————————————————————— #
    with open(f"{study_dir}/trials.log", "w") as f:
        f.write(
            f"Total trials: {len(study.trials)}\n"
            f"Pruned trials: {sum(t.state==TrialState.PRUNED for t in study.trials)}\n"
            f"Failed trials: {sum(t.state==TrialState.FAIL for t in study.trials)}\n"
        )

    # —————————————————————————— Generate Study Analysis ————————————————————————— #
    (clear(), analyze_study(study, table_dir=os.path.join(study_dir, "analysis")))
    
except Exception as e:
    print(f"\n An error occurred: {e}\n")
    traceback.print_exc()

    with open(os.path.join(logs_dir, "training_error.log"), "a") as f:
        f.write(f"An error occurred during training:\n{e}\n{traceback.format_exc()}\n\n")
finally:
    # Write success flag for the auto restart script
    Path("/tmp/success.flag").write_text("SUCCESS")

    # Clean up directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)



Analyzing study...
Generating summary tables...
Creating hyperparameter distribution plots...
Creating numeric parameters distribution plot (1 parameters)...
No categorical parameters found for distribution plotting.
Calculating parameter importances...
Analyzing Spearman correlations...
Creating boxplots for parameter distributions...
Creating numeric parameters boxplots (1 parameters)...
Performing trend analysis...
Creating optimal ranges analysis...
Analysis complete! All results saved to the specified directories.
